# Kaggle runner — HAM10000 experiments

**Setup (một lần cho mỗi notebook):**
1. *Add Input* → dataset `kmader/skin-cancer-mnist-ham10000`.
2. *Settings* → Accelerator **GPU T4 x2** hoặc **P100**; **Internet: On** (clone repo + tải trọng số ImageNet).
3. Sửa ô cấu hình bên dưới (danh sách thí nghiệm, seed), rồi *Save Version → Save & Run All (Commit)* để chạy nền.
4. Xong: tải `outputs.zip` ở tab *Output* về, giải nén vào `D:\DL\` (giữ cấu trúc `outputs/...`).

Test set **chỉ** được chấm khi `EVAL_TEST = True` cho cấu hình đã chốt (PLAN_BTL.md mục 0).

In [ ]:
# ===== CẤU HÌNH =====
REPO_URL = "https://github.com/duty00/Deep-Learning.git"   # repo đang public nên clone không cần token
BRANCH = "main"
EXPERIMENTS = ["e13_effb0_concat_la", "e10_effb0_concat_ce", "e11_effb0_film_la", "e09_film3_la",
               "e12e_concat_la_frac25", "e12f_concat_la_frac50", "e12a_concat_ce_frac25", "e12b_concat_ce_frac50"]   # tên file trong configs/experiments/ (không có .json)
SEEDS = [42, 43, 44]
MICRO_BATCH = 32          # GPU 16 GB: 32 (accum 1). Batch hiệu dụng vẫn = 32.
EVAL_TEST = False         # True CHỈ khi cấu hình đã chốt bằng validation
DEBUG = False             # True: chạy thử 2 epoch x 8 batch
N_GPUS = None             # None: dùng hết GPU thấy được (T4 x2 -> 2 run chạy song song)

In [ ]:
import os, subprocess, glob
os.chdir("/kaggle/working")
if not os.path.exists("DL"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "DL"], check=True)
os.chdir("/kaggle/working/DL")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
# Kaggle đã có torch/torchvision/timm; chỉ cài package của nhóm, không đụng phiên bản torch.
subprocess.run("pip install -q -e . --no-deps && pip install -q timm", shell=True, check=True)

In [ ]:
# Liên kết dataset Kaggle vào đúng đường dẫn config mong đợi (data/raw/ham10000).
candidates = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/HAM10000_metadata.csv", recursive=True)]
assert candidates, "Không thấy HAM10000_metadata.csv: hãy Add Input dataset kmader/skin-cancer-mnist-ham10000"
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/ham10000"):
    os.symlink(candidates[0], "data/raw/ham10000")
print("dataset:", candidates[0])
# Bắt buộc: xác nhận đúng file metadata + manifest đã commit (SHA-256, không rò rỉ lesion).
subprocess.run(["python", "scripts/make_splits.py", "--verify"], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import queue, threading, time

# Mỗi GPU chạy một run (train + evaluate) cùng lúc; log đầy đủ của từng run ở outputs/_logs/<run>.log.
n_gpus = N_GPUS or max(1, len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()))
jobs = queue.Queue()
for seed in SEEDS:                      # xếp theo seed trước: seed 42 của mọi thí nghiệm xong sớm nhất
    for exp in EXPERIMENTS:
        jobs.put((exp, seed))
os.makedirs("outputs/_logs", exist_ok=True)
failed, lock, t0 = [], threading.Lock(), time.time()

def say(msg):
    with lock:
        print(f"[{(time.time() - t0) / 60:6.1f} phút] {msg}", flush=True)

def run_job(gpu, exp, seed):
    cfg = f"configs/experiments/{exp}.json"
    extra = ["--seed", str(seed), "--micro-batch-size", str(MICRO_BATCH)] + (["--debug"] if DEBUG else [])
    name = f"{exp}-s{seed}" + ("-debug" if DEBUG else "")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    with open(f"outputs/_logs/{name}.log", "w", encoding="utf-8") as log:
        def call(args):
            log.write(">>> " + " ".join(args) + chr(10)); log.flush()
            subprocess.run(["python", "-m", "dl_project"] + args, check=True, env=env, stdout=log, stderr=subprocess.STDOUT)
        call(["train", "--config", cfg] + extra)
        best = sorted(glob.glob(f"outputs/{name}/train/*/best.pt"))[-1]
        call(["evaluate", "--config", cfg, "--checkpoint", best, "--split", "validation"] + extra)
        if EVAL_TEST:
            call(["evaluate", "--config", cfg, "--checkpoint", best, "--split", "test"] + extra)
    return name

def worker(gpu):
    while True:
        try:
            exp, seed = jobs.get_nowait()
        except queue.Empty:
            return
        name = f"{exp}-s{seed}"
        say(f"GPU{gpu} bắt đầu {name}")
        try:
            run_job(gpu, exp, seed)
            summary = sorted(glob.glob(f"outputs/{name}*/evaluate/*/train.log"))
            last = open(summary[-1], encoding="utf-8").read().strip().splitlines()[-1] if summary else ""
            say(f"GPU{gpu} xong {name} | {last}")
        except Exception as exc:
            with lock:
                failed.append(name)
            say(f"GPU{gpu} LỖI {name}: {exc} (xem outputs/_logs/{name}.log)")

print(f"{jobs.qsize()} run trên {n_gpus} GPU")
threads = [threading.Thread(target=worker, args=(g,)) for g in range(n_gpus)]
for t in threads: t.start()
for t in threads: t.join()
say("LỖI: " + ", ".join(failed) if failed else "Tất cả run thành công.")

In [ ]:
subprocess.run(["python", "scripts/aggregate_results.py", "--split", "validation"], check=True)
print(open("reports/tables/results_validation.csv").read())
# Đóng gói kết quả (gồm checkpoint) để tải về máy.
subprocess.run("cd /kaggle/working/DL && zip -qr /kaggle/working/outputs.zip outputs reports/tables", shell=True, check=True)
print("-> /kaggle/working/outputs.zip")